# **Interacción Radiación-Materia — Hoja 4 de problemas**
### Desintegraciones nucleares: cadenas radiactivas, activación neutrónica y datación

Este cuaderno contiene la resolución detallada de la **Hoja 4** de problemas. Cada problema incluye:

* El **enunciado**.
* La **resolución razonada** en Markdown, con las ecuaciones necesarias.
* Una celda de **cálculo en Python** que reproduce los resultados numéricos.
* En muchos casos, **gráficas** y **widgets interactivos** para explorar la física del problema (equilibrios radiactivos, generadores de $^{99m}$Tc, activación neutrónica, datación...).

### **Preparativos**
La siguiente celda ha de ejecutarse para cargar las librerías y datos necesarios para el correcto funcionamiento de este cuaderno

In [ ]:
import os

# 1. Definir los datos del repositorio
USUARIO_GH = "nietootein"
REPOSITORIO_GH = "nirma"

# 2. Resetear siempre al directorio base ("home") de Google Colab
directorio_home = '/content'
if os.path.exists(directorio_home):
    os.chdir(directorio_home)
print(f"🏠 Partiendo del directorio de inicio: {os.getcwd()}")

# 3. Clonar el repositorio solo si no existe (y si no estamos ya dentro de él)
if os.path.basename(os.getcwd()) == REPOSITORIO_GH:
    print(f"ℹ️ Ya estamos dentro del repositorio '{REPOSITORIO_GH}'.")
elif os.path.basename(os.getcwd()) == 'notebooks' and os.path.exists('../data'):
    os.chdir('..')
    print("ℹ️ Ejecución local detectada: subiendo a la raíz del repositorio.")
else:
    if not os.path.exists(REPOSITORIO_GH):
        print(f"📦 Clonando el repositorio '{REPOSITORIO_GH}'...")
        !git clone https://github.com/{USUARIO_GH}/{REPOSITORIO_GH}.git
    else:
        print(f"ℹ️ El repositorio '{REPOSITORIO_GH}' ya está disponible en el directorio base.")
    os.chdir(REPOSITORIO_GH)
print(f"📂 Directorio de trabajo activo establecido en: {os.getcwd()}")

# 4. Importar librerías de Python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display
from scipy.optimize import brentq

sns.set_theme(style='whitegrid', palette='colorblind', font='DejaVu Sans')

#### **Constantes y funciones auxiliares comunes**
Las dos herramientas que más usaremos en esta hoja son:

1. La **ley de desintegración**: $N(t) = N_0 e^{-\lambda t}$, $A = \lambda N$, con $\lambda = \ln 2/T_{1/2}$.
2. La **solución de Bateman** para una cadena de dos miembros A → B (B inicialmente ausente):
$$ A_B(t) = A_A(0)\,\frac{\lambda_B}{\lambda_B - \lambda_A}\left(e^{-\lambda_A t} - e^{-\lambda_B t}\right) $$

In [ ]:
# =====================================================================
# CONSTANTES
# =====================================================================
NA = 6.022e23              # mol^-1
Ci = 3.7e10                # Bq
eV = 1.602176634e-19       # J
barn = 1e-24               # cm^2
dia = 86400.0              # s
anio = 365.25 * dia        # s

# =====================================================================
# FUNCIONES AUXILIARES
# =====================================================================
def lam(T):
    # Constante de desintegración a partir del periodo (mismas unidades de tiempo inversas)
    return np.log(2) / T

def bateman_hija(A_madre_0, lA, lB, t):
    # Actividad de la hija B (inicialmente ausente) en la cadena A -> B
    if np.isclose(lA, lB, rtol=1e-9, atol=0.0):
        return A_madre_0 * lB * t * np.exp(-lA * t)
    return A_madre_0 * lB / (lB - lA) * (np.exp(-lA * t) - np.exp(-lB * t))

print("✅ Constantes y funciones auxiliares cargadas.")

### **Problema 1**
El $^{238}$Pu emite partículas $\alpha$ de energía media 5,49 MeV y periodo 86,4 años. Su hijo, el $^{234}$U, tiene un periodo superior a $10^5$ años. Una masa total de 23,2 kg de $^{238}$Pu fue utilizada en la nave Cassini para conseguir mediante generadores termoeléctricos la energía necesaria para sus sistemas electrónicos. Hallar la potencia térmica (en kW) que generaba esta fuente radiactiva en el momento de su lanzamiento (Octubre 1997) y 15 años después.

---

#### **1. Actividad inicial**
$$ \lambda = \frac{\ln 2}{86.4\text{ años}} = 2.54\times10^{-10}\text{ s}^{-1}, \qquad N = \frac{N_A}{P_a}\,m = \frac{6.02\times10^{23}}{238\text{ g}}\times23.2\times10^3\text{ g} = 5.87\times10^{25}\text{ núcleos} $$
$$ A = \lambda N = 1.49\times10^{16}\text{ s}^{-1} \;(\approx 4\times10^5\text{ Ci}) $$

#### **2. Potencia térmica**
Las partículas $\alpha$ se frenan en el propio material, depositando toda su energía:
$$ P = A\,E_\alpha = 1.49\times10^{16}\text{ s}^{-1}\times5.49\times10^6\text{ eV}\times1.602\times10^{-19}\text{ J/eV} \approx \mathbf{13.1\text{ kW}} $$

#### **3. Potencia 15 años después**
La hija $^{234}$U tiene un periodo tan largo que su contribución es despreciable (ver más abajo), así que la potencia decae con el periodo del $^{238}$Pu:
$$ P(15\text{ años}) = 13.1\text{ kW}\times e^{-\ln2\cdot15/86.4} \approx \mathbf{11.6\text{ kW}} $$

#### **Correcciones de orden menor**
* **Energía de retroceso del núcleo hijo.** La energía liberada en la desintegración es $Q = E_\alpha\left(1 + m_\alpha/m_U\right) = 1.017\,E_\alpha$. El núcleo de $^{234}$U también deposita su energía de retroceso (93 keV), así que la potencia real es $P \approx 13.1\times1.017 \approx 13.35$ kW.
* **Contribución del $^{234}$U** ($T_B = 2.46\times10^5$ años, $E_\alpha = 4.8$ MeV). Para $t \ll T_B$: $A_B/A_A \approx \lambda_B t = 4.2\times10^{-5}$ a los 15 años; con el cociente de energías (0.87), $P_U/P_{Pu} \approx 3.7\times10^{-5}$ (el cálculo exacto con Bateman da $3.9\times10^{-5}$, porque la actividad del Pu también ha disminuido): totalmente despreciable. Las actividades sólo se igualarían al cabo de $\sim 1000$ años.

> **Nota práctica:** los generadores termoeléctricos (RTG) convierten en electricidad sólo un ~6 % de esta potencia térmica: unos 800 W eléctricos al lanzamiento para toda la nave.

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
m_Pu = 23.2e3          # g
Pa_Pu = 238.0          # g/mol
T_Pu = 86.4 * anio     # s
E_a = 5.49e6           # eV
T_U = 2.46e5 * anio    # s (234U)
E_aU = 4.8e6           # eV

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
l_Pu, l_U = lam(T_Pu), lam(T_U)
N_Pu = NA / Pa_Pu * m_Pu
A0_Pu = l_Pu * N_Pu
P0 = A0_Pu * E_a * eV
P15 = P0 * np.exp(-l_Pu * 15 * anio)
factor_retroceso = 1 + 4 / 234

def potencia_total(t):
    A_Pu = A0_Pu * np.exp(-l_Pu * t)
    A_U = bateman_hija(A0_Pu, l_Pu, l_U, t)
    return A_Pu * E_a * eV, A_U * E_aU * eV

t_igual = np.log(l_U / l_Pu) / (l_U - l_Pu)

# =====================================================================
# 3. IMPRESIÓN DE RESULTADOS
# =====================================================================
print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 1)                    ")
print("=====================================================================")
print(f"λ = {l_Pu:.3e} s⁻¹,  N = {N_Pu:.3e} núcleos,  A = {A0_Pu:.3e} s⁻¹ = {A0_Pu/Ci:.2e} Ci")
print(f"Potencia térmica al lanzamiento:    P = {P0/1e3:.1f} kW   (con retroceso: {P0*factor_retroceso/1e3:.2f} kW)")
print(f"Potencia térmica 15 años después:   P = {P15/1e3:.1f} kW")
PPu15, PU15 = potencia_total(15 * anio)
print(f"Contribución del 234U a los 15 años: P_U/P_Pu = {PU15/PPu15:.1e}")
print(f"Las actividades de 238Pu y 234U se igualan a los {t_igual/anio:.0f} años")
print("=====================================================================")

#### **Bonus: La "pila" radiactiva de una sonda espacial**

El widget muestra la evolución de la potencia térmica del generador de $^{238}$Pu a lo largo de la misión (y mucho más allá), separando las contribuciones del $^{238}$Pu y de su hija $^{234}$U. Puedes elegir otros radioisótopos candidatos para un RTG.

**Cuestiones:**
1. La misión Cassini terminó en septiembre de 2017. ¿Qué potencia térmica tenía entonces el generador?
2. ¿Por qué no se usa $^{210}$Po ($T = 138$ días, $E_\alpha = 5.3$ MeV), que tiene una potencia específica enorme? ¿Y por qué no el $^{241}$Am ($T = 433$ años)?
3. ¿Cuánto tiempo tiene que pasar para que la potencia del $^{234}$U sea comparable a la del $^{238}$Pu? ¿Tiene interés práctico?

In [ ]:
isotopos_RTG = {
    #             T (años), E_alpha (MeV), Pa (g/mol)
    '238Pu':      (86.4,    5.49,  238),
    '210Po':      (138/365.25, 5.30, 210),
    '241Am':      (433,     5.49,  241),
    '244Cm':      (18.1,    5.80,  244),
}

def rtg(isotopo, masa_kg, anios):
    T, Ea, Pa = isotopos_RTG[isotopo]
    l = lam(T * anio)
    N0 = NA / Pa * masa_kg * 1e3
    t = np.linspace(0, anios, 600) * anio
    P = l * N0 * np.exp(-l * t) * Ea * 1e6 * eV / 1e3   # kW
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    ax = axes[0]
    ax.plot(t / anio, P, color='#c44e52', lw=2.5)
    if isotopo == '238Pu':
        ax.axvline(20, color='gray', ls=':'); ax.text(20.5, P[0] * 0.95, 'Fin de Cassini (2017)', color='gray')
    ax.set_xlabel('Años desde el lanzamiento'); ax.set_ylabel('Potencia térmica (kW)')
    ax.set_ylim(0, P[0] * 1.05)
    ax.set_title(f'{masa_kg:.1f} kg de {isotopo}: P₀ = {P[0]:.2f} kW ({P[0]/masa_kg:.2f} W/g)', fontweight='bold')
    ax = axes[1]
    tt = np.logspace(-1, 7, 600) * anio
    PPu, PU = potencia_total(tt)
    ax.loglog(tt / anio, PPu / 1e3, color='#c44e52', lw=2.5, label='²³⁸Pu')
    ax.loglog(tt / anio, PU / 1e3, color='#4c72b0', lw=2.5, label='²³⁴U (hija)')
    ax.axvline(t_igual / anio, color='gray', ls=':')
    ax.set_ylim(1e-6, 30); ax.set_xlabel('Años'); ax.set_ylabel('Potencia térmica (kW)')
    ax.set_title('Cadena ²³⁸Pu → ²³⁴U (23.2 kg iniciales)', fontweight='bold'); ax.legend()
    plt.tight_layout(); plt.show()
    print(f"Potencia a los {anios:.0f} años: {P[-1]:.3f} kW ({100*P[-1]/P[0]:.1f}% de la inicial)")

interact(rtg,
         isotopo=widgets.Dropdown(options=list(isotopos_RTG), value='238Pu', description='Isótopo'),
         masa_kg=widgets.FloatSlider(value=23.2, min=0.1, max=50, step=0.1, description='Masa (kg)',
                                     continuous_update=False),
         anios=widgets.FloatSlider(value=40, min=1, max=300, step=1, description='Años',
                                   continuous_update=False));

### **Problema 2**
Una muestra radiactiva contiene originalmente ($t = 0$) 1 mCi de $^{191}$Os. Usando los datos de la desintegración del $^{191}$Os que se muestran más abajo, resolver las siguientes cuestiones:

**a)** Hallar la masa de $^{191}$Os presente en la muestra en $t = 0$.

**b)** Hallar la actividad de $^{191m}$Ir de la muestra en el instante $t = 12$ días.

**c)** Hallar el número de partículas beta emitidas por la muestra en el intervalo de tiempo 10 días - 12 días.

**d)** Hallar el número total de rayos $\gamma$ emitidos por la muestra hasta que cese su actividad.

$$ ^{191}\text{Os} \xrightarrow{\beta^-} {}^{191m}\text{Ir} \xrightarrow{\gamma} {}^{191}\text{Ir} $$

Periodos: $T(^{191}\text{Os}) = 15,4$ días, $T(^{191m}\text{Ir}) = 4,94$ segundos.

---

#### **a) Masa de $^{191}$Os**
$$ \lambda = \frac{\ln 2}{15.4\text{ d}} = 5.21\times10^{-7}\text{ s}^{-1}, \qquad N = \frac{A}{\lambda} = \frac{3.7\times10^7\text{ s}^{-1}}{5.21\times10^{-7}\text{ s}^{-1}} = 7.10\times10^{13}\text{ núcleos} $$
$$ m = \frac{N}{N_A}\,P_a = \frac{7.10\times10^{13}}{6.02\times10^{23}}\times191\text{ g} = \mathbf{2.25\times10^{-8}\text{ g}} \;(22.5\text{ ng}) $$
¡Un milicurio de un isótopo de periodo de unas semanas son apenas unas decenas de nanogramos!

#### **b) Actividad de $^{191m}$Ir a los 12 días**
Como $T(^{191}\text{Os}) \gg T(^{191m}\text{Ir})$ (15.4 días frente a 5 s), se alcanza el **equilibrio secular** en unos segundos: la hija tiene en todo momento la misma actividad que la madre.
$$ A_{Ir^m}(12\text{ d}) = A_{Os}(12\text{ d}) = 1\text{ mCi}\times e^{-\ln2\cdot12/15.4} = \mathbf{0.583\text{ mCi}} $$

#### **c) Partículas $\beta$ emitidas entre los días 10 y 12**
Cada desintegración del $^{191}$Os emite una $\beta^-$, así que el número de $\beta$ es el número de núcleos de Os desintegrados en ese intervalo:
$$ N_\beta = N_0\left(e^{-\lambda\cdot10\text{ d}} - e^{-\lambda\cdot12\text{ d}}\right) = 7.10\times10^{13}\,(0.6376 - 0.5827) = \mathbf{3.90\times10^{12}} $$

#### **d) Número total de rayos $\gamma$**
Cada núcleo de Os acaba produciendo un $^{191m}$Ir, que se desexcita emitiendo un $\gamma$. El número total de $\gamma$ es el número inicial de núcleos:
$$ N_\gamma = N_0 = \mathbf{7.10\times10^{13}} $$
(En realidad esta transición de 171 keV está fuertemente **convertida internamente**, de modo que una parte importante de las desexcitaciones produce electrones de conversión en lugar de rayos $\gamma$; el enunciado lo ignora).

> **Nota:** en las soluciones de clase se emplea $1\text{ Ci} = 3.65\times10^{10}$ s⁻¹, lo que da $N_0 = 7.01\times10^{13}$, $m = 2.22\times10^{-8}$ g y $N_\beta = 3.85\times10^{12}$. Aquí usamos el valor correcto, $1\text{ Ci} = 3.7\times10^{10}$ s⁻¹.

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
A0_Os = 1e-3 * Ci          # Bq
T_Os = 15.4 * dia
T_Ir = 4.94                # s
Pa_Os = 191.0

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
l_Os, l_Ir = lam(T_Os), lam(T_Ir)
N0_Os = A0_Os / l_Os
m_Os = N0_Os / NA * Pa_Os
A_Ir_12 = bateman_hija(A0_Os, l_Os, l_Ir, 12 * dia)
N_beta = N0_Os * (np.exp(-l_Os * 10 * dia) - np.exp(-l_Os * 12 * dia))

# =====================================================================
# 3. IMPRESIÓN DE RESULTADOS
# =====================================================================
print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 2)                    ")
print("=====================================================================")
print(f"a) λ = {l_Os:.3e} s⁻¹,  N₀ = {N0_Os:.3e},  m = {m_Os:.3e} g")
print(f"b) A(191mIr, 12 d) = {A_Ir_12/Ci*1e3:.3f} mCi   (Bateman exacta)")
print(f"c) N_β (10-12 d) = {N_beta:.3e}")
print(f"d) N_γ total = {N0_Os:.3e}")
print("=====================================================================")

# Gráfica: establecimiento del equilibrio secular (primeros segundos) y evolución a largo plazo
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
t = np.linspace(0, 40, 400)
axes[0].plot(t, np.full_like(t, A0_Os) / Ci * 1e3, color='#4c72b0', lw=2.5, label='¹⁹¹Os')
axes[0].plot(t, bateman_hija(A0_Os, l_Os, l_Ir, t) / Ci * 1e3, color='#c44e52', lw=2.5, label='¹⁹¹ᵐIr')
axes[0].set_xlabel('t (s)'); axes[0].set_ylabel('Actividad (mCi)')
axes[0].set_title('Primeros segundos: se alcanza el equilibrio secular', fontweight='bold'); axes[0].legend()
t = np.linspace(0, 60, 400) * dia
axes[1].plot(t / dia, A0_Os * np.exp(-l_Os * t) / Ci * 1e3, color='#4c72b0', lw=3, label='¹⁹¹Os')
axes[1].plot(t / dia, bateman_hija(A0_Os, l_Os, l_Ir, t) / Ci * 1e3, color='#c44e52', lw=1.5, ls='--', label='¹⁹¹ᵐIr')
axes[1].axvspan(10, 12, color='gold', alpha=0.4, label='Intervalo del apartado c)')
axes[1].set_xlabel('t (días)'); axes[1].set_ylabel('Actividad (mCi)')
axes[1].set_title('Evolución a largo plazo', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()

### **Problema 3**
El espectro alfa de una muestra de $^{241}$Am revela la presencia de líneas procedentes del $^{237}$Np con una intensidad de $3,2\cdot10^{-4}$ % respecto de la intensidad de las líneas del $^{241}$Am.

**a)** ¿Qué antigüedad tiene dicha muestra suponiendo que originalmente era $^{241}$Am puro?

**b)** Si la actividad alfa de la muestra es de 1 Curio, ¿cuál será la actividad beta (en la misma época de dicho experimento)?

$$ ^{241}\text{Am} \xrightarrow{\alpha} {}^{237}\text{Np} \xrightarrow{\alpha} {}^{233}\text{Pa} \xrightarrow{\beta^-} {}^{233}\text{U} $$

Periodos: $T(^{241}\text{Am}) = 433$ años, $T(^{237}\text{Np}) = 2,14\cdot10^6$ años, $T(^{233}\text{Pa}) = 27$ días.

---

#### **a) Antigüedad de la muestra**
El cociente de intensidades de las líneas $\alpha$ es el cociente de actividades: $A_2/A_1 = 3.2\times10^{-6}$. Entre $^{241}$Am (1) y $^{237}$Np (2) **no hay equilibrio** ($T_2 \gg T_1$). Con la solución de Bateman:
$$ \frac{A_2(t)}{A_1(t)} = \frac{\lambda_2}{\lambda_2 - \lambda_1}\left(1 - e^{-(\lambda_2 - \lambda_1)t}\right) = \frac{T_1}{T_1 - T_2}\left(1 - e^{-(\lambda_2-\lambda_1)t}\right) $$
Despejando $t$ (y usando $\lambda_2 \ll \lambda_1$):
$$ t = \frac{T_1}{\ln 2}\ln\left(1 + \frac{A_2}{A_1}\frac{T_2}{T_1}\right) = \frac{433\text{ años}}{\ln 2}\ln\left(1 + 3.2\times10^{-6}\times\frac{2.14\times10^6}{433}\right) = \mathbf{9.80\text{ años}} $$
(Como $t \ll T_1$, también se podría aproximar $A_2/A_1 \approx \lambda_2 t$, que da $t \approx 9.88$ años.)

#### **b) Actividad beta**
La actividad $\beta$ es la del $^{233}$Pa. Entre $^{237}$Np y $^{233}$Pa ($T = 27$ d $\ll 2.14\times10^6$ años) hay **equilibrio secular**: $A_{Pa} = A_{Np}$. La actividad $\alpha$ total es prácticamente la del Am, así que:
$$ A_\beta = A_{Np} = 1\text{ Ci}\times3.2\times10^{-6} = 3.2\ \mu\text{Ci} = \mathbf{1.2\times10^5\text{ Bq}} $$

> El método funciona como **reloj**: cuanto más antigua es la muestra, mayores son las líneas del Np respecto de las del Am (ver widget).

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
T1, T2, T3 = 433.0, 2.14e6, 27 / 365.25   # años
cociente = 3.2e-6
A_alfa = 1 * Ci

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
l1, l2, l3 = lam(T1), lam(T2), lam(T3)
t_exacta = brentq(lambda t: l2 / (l2 - l1) * (1 - np.exp(-(l2 - l1) * t)) - cociente, 1e-3, 1e3)
t_aprox = T1 / np.log(2) * np.log(1 + cociente * T2 / T1)
t_lineal = cociente / l2
A_beta = cociente * A_alfa

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 3)                    ")
print("=====================================================================")
print(f"a) Antigüedad: t = {t_exacta:.2f} años (exacta)  |  {t_aprox:.2f} años (fórmula)  |  {t_lineal:.2f} (A₂/A₁ ≈ λ₂t)")
print(f"b) Actividad β = A(Np) = {A_beta:.2e} Bq = {A_beta/Ci*1e6:.1f} µCi")
print("=====================================================================")

t = np.logspace(-1, 4, 500)
A1 = np.exp(-l1 * t)
A2 = bateman_hija(1.0, l1, l2, t)
fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(t, A2 / A1, color='#4c72b0', lw=2.5)
ax.plot(t_exacta, cociente, '*', color='gold', ms=18, mec='k', label=f'Muestra: {t_exacta:.1f} años')
ax.axhline(cociente, color='gray', ls=':')
ax.set_xlabel('Antigüedad (años)'); ax.set_ylabel('A(²³⁷Np) / A(²⁴¹Am)')
ax.set_title('"Reloj" ²⁴¹Am → ²³⁷Np: cociente de intensidades de las líneas α', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

#### **Bonus: Laboratorio de equilibrios radiactivos (Problemas 2, 3 y 4)**

En una cadena A → B con B inicialmente ausente, el comportamiento depende del cociente de periodos:

| Situación | Condición | Comportamiento | Ejemplo de la hoja |
| :--- | :--- | :--- | :--- |
| **Equilibrio secular** | $T_A \gg T_B$ | $A_B \to A_A$ | $^{191}$Os → $^{191m}$Ir; $^{237}$Np → $^{233}$Pa |
| **Equilibrio transitorio** | $T_A > T_B$ | $A_B/A_A \to T_A/(T_A - T_B) > 1$ | $^{99}$Mo → $^{99m}$Tc |
| **No equilibrio** | $T_A < T_B$ | la hija acaba sobreviviendo a la madre | $^{241}$Am → $^{237}$Np; $^{238}$Pu → $^{234}$U |

En los tres casos, la actividad de la hija alcanza su **máximo** cuando $A_B = A_A$, en
$$ t_{max} = \frac{\ln(\lambda_B/\lambda_A)}{\lambda_B - \lambda_A} $$

**Cuestiones:**
1. Con los datos del $^{99}$Mo → $^{99m}$Tc ($T_A = 66$ h, $T_B = 6$ h), ¿cuánto vale el cociente $A_B/A_A$ en equilibrio? ¿Cuándo es máxima la actividad del Tc?
2. ¿Qué ocurre si $T_A = T_B$?
3. En el caso de no equilibrio, ¿qué pendiente tiene $\log A_B$ a tiempos largos?

In [ ]:
def laboratorio_equilibrios(TA, TB, escala_log):
    lA, lB = lam(TA), lam(TB)
    t_max_plot = 8 * max(TA, TB)
    t = np.linspace(0, t_max_plot, 1000)
    AA = np.exp(-lA * t)
    AB = bateman_hija(1.0, lA, lB, t)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    ax = axes[0]
    ax.plot(t, AA, color='#4c72b0', lw=2.5, label=f'Madre A (T = {TA:g})')
    ax.plot(t, AB, color='#c44e52', lw=2.5, label=f'Hija B (T = {TB:g})')
    ax.plot(t, AA + AB, color='k', ls=':', lw=1.5, label='Total')
    if not np.isclose(lA, lB):
        tm = np.log(lB / lA) / (lB - lA)
        ax.axvline(tm, color='gray', ls='--'); ax.text(tm, 1.02, f' t_max = {tm:.3g}', color='gray')
    if escala_log:
        ax.set_yscale('log'); ax.set_ylim(1e-4, 3)
    ax.set_xlabel('Tiempo (mismas unidades que los periodos)'); ax.set_ylabel('Actividad / A_A(0)')
    ax.set_title('Actividades de madre e hija', fontweight='bold'); ax.legend()
    ax = axes[1]
    ax.plot(t[1:], AB[1:] / AA[1:], color='#8172b3', lw=2.5)
    if TA > TB:
        ax.axhline(TA / (TA - TB), color='gray', ls='--', label=f'T_A/(T_A−T_B) = {TA/(TA-TB):.3f}')
        regimen = 'EQUILIBRIO SECULAR' if TA > 50 * TB else 'EQUILIBRIO TRANSITORIO'
    else:
        regimen = 'NO HAY EQUILIBRIO'
        ax.set_yscale('log')
    ax.set_xlabel('Tiempo'); ax.set_ylabel('A_B / A_A')
    ax.set_title(f'Cociente de actividades: {regimen}', fontweight='bold')
    if TA > TB: ax.legend()
    plt.tight_layout(); plt.show()

interact(laboratorio_equilibrios,
         TA=widgets.FloatLogSlider(value=66, base=10, min=-1, max=3, step=0.02, description='T madre',
                                   continuous_update=False),
         TB=widgets.FloatLogSlider(value=6, base=10, min=-1, max=3, step=0.02, description='T hija',
                                   continuous_update=False),
         escala_log=widgets.Checkbox(value=False, description='Escala logarítmica'));

### **Problema 4**
El $^{99}$Mo se desintegra por emisión $\beta$ quedando convertido en un estado metaestable del $^{99}$Tc que se desexcita cayendo al estado fundamental emitiendo un rayo $\gamma$ de 140 keV o un electrón de conversión. En el dibujo se dan los datos de la cadena de desintegración indicándose los periodos. Originalmente se dispone de una fuente de $^{99}$Mo en equilibrio secular con $^{99m}$Tc cuya actividad es de $3,7\times10^7$ des. s⁻¹. En ese momento se separa instantáneamente el tecnecio de la fuente de molibdeno ($t = 0$).

**a)** Hallar la actividad del $^{99m}$Tc en la fuente de tecnecio puro y en la de molibdeno a las 5 h de la separación.

**b)** Hallar la energía e intensidad (partículas por segundo) de las líneas $\gamma$ y de electrones de conversión de la fuente de $^{99m}$Tc en ese instante ($t$ = 5 h). Representar gráficamente el espectro.

**DATOS:** Las energías de las capas K y L del tecnecio y los coeficientes de conversión interna son: $E_K$ = 21 keV, $E_L$ = 3 keV, $\alpha_K$ = 0,101, $\alpha_L$ = 0,022.

$^{99}$Mo ($T$ = 66 h) $\xrightarrow{\beta}$ $^{99m}$Tc ($T$ = 6.0 h) $\xrightarrow{140\text{ keV}}$ $^{99}$Tc ($T > 10^5$ años)

---

#### **a) Actividades a las 5 h**
* **Fuente de Tc puro:** el Tc separado simplemente decae:
$$ A_{Tc}(5\text{ h}) = A_0\,e^{-\ln2\cdot5/6} = 3.7\times10^7\times0.561 = \mathbf{2.08\times10^7\text{ s}^{-1}} $$
* **Tc regenerado en la fuente de Mo:** tras la separación, el Mo vuelve a producir Tc (solución de Bateman con $A_{Mo}(0) = A_0$):
$$ A_{Tc}^{Mo}(5\text{ h}) = A_0\frac{T_{Mo}}{T_{Mo} - T_{Tc}}\left(e^{-\ln2\cdot t/T_{Mo}} - e^{-\ln2\cdot t/T_{Tc}}\right) = 3.7\times10^7\times1.1\times(0.949 - 0.561) = \mathbf{1.58\times10^7\text{ s}^{-1}} $$
*Aproximación:* tratando la actividad del Mo como casi constante, $A_{Tc}^{Mo} \approx A_{Mo}(t)\left(1 - e^{-\lambda_{Tc}t}\right) = A_0\times0.95\times0.44 = 1.54\times10^7$ s⁻¹.

A las 5 h, el generador ya ha recuperado más de la mitad de la actividad de Tc inicial. Éste es el principio del **generador de $^{99m}$Tc** ("vaca de molibdeno"), la fuente de radioisótopos más utilizada en medicina nuclear: el Tc se "ordeña" (eluye) periódicamente del Mo.

#### **b) Espectro de la fuente de $^{99m}$Tc a las 5 h**
La transición de 140 keV puede emitir un $\gamma$ o un electrón de conversión. Con el coeficiente de conversión total $\alpha_T = \alpha_K + \alpha_L = 0.123$ (cociente entre las probabilidades de conversión y de emisión $\gamma$):
$$ P_\gamma = \frac{1}{1+\alpha_T} = 0.890, \qquad P_{e_K} = \frac{\alpha_K}{1+\alpha_T} = 0.0899, \qquad P_{e_L} = \frac{\alpha_L}{1+\alpha_T} = 0.0196 $$

| Línea | Energía | Intensidad ($A_{Tc} = 2.08\times10^7$ s⁻¹) |
| :--- | :---: | :---: |
| $\gamma$ | 140 keV | $1.85\times10^7$ γ/s |
| Electrones de conversión K | $140 - 21 = 119$ keV | $1.87\times10^6$ e⁻/s |
| Electrones de conversión L | $140 - 3 = 137$ keV | $4.1\times10^5$ e⁻/s |

Tras cada conversión interna queda una vacante en la capa K o L, que se llena emitiendo **rayos X característicos** del Tc (~18-21 keV) o **electrones Auger** (no pedidos en el problema).

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
A0_gen = 3.7e7                 # s^-1
T_Mo, T_Tc = 66.0, 6.0         # h
t5 = 5.0                       # h
E_g, E_K, E_L = 140.0, 21.0, 3.0
aK, aL = 0.101, 0.022

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
l_Mo, l_Tc = lam(T_Mo), lam(T_Tc)
A_Tc_puro = A0_gen * np.exp(-l_Tc * t5)
A_Tc_Mo = bateman_hija(A0_gen, l_Mo, l_Tc, t5)
A_Tc_Mo_aprox = A0_gen * np.exp(-l_Mo * t5) * (1 - np.exp(-l_Tc * t5))
aT = aK + aL
lineas = {'γ':           (E_g,       1 / (1 + aT)),
          'e⁻ conv. K':  (E_g - E_K, aK / (1 + aT)),
          'e⁻ conv. L':  (E_g - E_L, aL / (1 + aT))}

# =====================================================================
# 3. IMPRESIÓN DE RESULTADOS
# =====================================================================
print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 4)                    ")
print("=====================================================================")
print(f"a) A(99mTc) fuente de Tc puro (5 h):   {A_Tc_puro:.3e} s⁻¹")
print(f"   A(99mTc) en la fuente de Mo (5 h):  {A_Tc_Mo:.3e} s⁻¹  (aproximación: {A_Tc_Mo_aprox:.3e})")
print("-" * 69)
print(f"b) α_T = {aT:.3f}")
for nombre, (E, P) in lineas.items():
    print(f"   {nombre:<12}: E = {E:5.0f} keV,  P = {P:.4f},  I = {P*A_Tc_puro:.3e} partículas/s")
print("=====================================================================")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
t = np.linspace(0, 24, 400)
ax.plot(t, A0_gen * np.exp(-l_Tc * t), color='#4c72b0', lw=2.5, label='⁹⁹ᵐTc separado (fuente de Tc)')
ax.plot(t, bateman_hija(A0_gen, l_Mo, l_Tc, t), color='#c44e52', lw=2.5, label='⁹⁹ᵐTc regenerado (fuente de Mo)')
ax.plot(t, A0_gen * np.exp(-l_Mo * t), color='k', ls=':', label='⁹⁹Mo')
ax.axvline(t5, color='gray', ls='--')
ax.set_xlabel('Tiempo tras la separación (h)'); ax.set_ylabel('Actividad (s⁻¹)')
ax.set_title('a) Separación del tecnecio', fontweight='bold'); ax.legend()
ax = axes[1]
for (nombre, (E, P)), color in zip(lineas.items(), ['#4c72b0', '#c44e52', '#c44e52']):
    ax.vlines(E, 0, P * A_Tc_puro, color=color, lw=4)
    ax.text(E, P * A_Tc_puro * 1.3, f'{nombre}\n{E:.0f} keV', ha='center', fontsize=9)
ax.set_yscale('log'); ax.set_ylim(1e5, 1e8); ax.set_xlim(100, 160)
ax.set_xlabel('Energía (keV)'); ax.set_ylabel('Intensidad (partículas/s)')
ax.set_title('b) Espectro de la fuente de ⁹⁹ᵐTc a las 5 h (azul: γ; rojo: e⁻)', fontweight='bold')
plt.tight_layout(); plt.show()

#### **Bonus: El generador de $^{99m}$Tc en un hospital**

Los hospitales reciben semanalmente un generador de $^{99}$Mo y extraen ("eluyen") el $^{99m}$Tc varias veces. Este widget simula una semana de uso con eluciones periódicas: tras cada elución el Tc del generador cae a cero y vuelve a crecer.

**Cuestiones:**
1. ¿Cada cuántas horas conviene eluir para obtener la máxima actividad de Tc en cada extracción? (Pista: $t_{max}$ del laboratorio de equilibrios).
2. ¿Cuánta actividad total de Tc se obtiene en la semana con eluciones cada 24 h? ¿Y cada 6 h?
3. ¿Por qué el generador se reemplaza cada semana aproximadamente?

In [ ]:
def generador_Tc(A_Mo_inicial_GBq, intervalo_h, dias):
    t = np.linspace(0, dias * 24, 4000)
    A_Mo = A_Mo_inicial_GBq * np.exp(-l_Mo * t)
    A_Tc = np.zeros_like(t)
    elucion_t, elucion_A = [], []
    t_ultima = 0.0
    A_Mo_ultima = A_Mo_inicial_GBq        # actividad del Mo en la última elución
    A_Tc_ultima = A_Mo_inicial_GBq        # inicialmente en equilibrio (secular aprox.)
    for i, ti in enumerate(t):
        if ti - t_ultima >= intervalo_h:
            A_antes = A_Tc[i - 1]
            elucion_t.append(ti); elucion_A.append(A_antes)
            t_ultima, A_Mo_ultima, A_Tc_ultima = ti, A_Mo[i], 0.0
        dt = ti - t_ultima
        A_Tc[i] = bateman_hija(A_Mo_ultima, l_Mo, l_Tc, dt) + A_Tc_ultima * np.exp(-l_Tc * dt)
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(t, A_Mo, color='k', ls=':', lw=1.5, label='⁹⁹Mo')
    ax.plot(t, A_Tc, color='#c44e52', lw=2, label='⁹⁹ᵐTc en el generador')
    ax.vlines(elucion_t, 0, elucion_A, color='#4c72b0', lw=3, label='Actividad eluida')
    ax.set_xlabel('Tiempo (h)'); ax.set_ylabel('Actividad (GBq)')
    ax.set_title(f'Generador de ⁹⁹ᵐTc: eluciones cada {intervalo_h:.0f} h', fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"Número de eluciones: {len(elucion_t)}   |   Actividad total eluida: {sum(elucion_A):.1f} GBq   |   "
          f"Media por elución: {np.mean(elucion_A) if elucion_A else 0:.1f} GBq")

interact(generador_Tc,
         A_Mo_inicial_GBq=widgets.FloatSlider(value=40, min=5, max=100, step=1, description='A(Mo) (GBq)',
                                              continuous_update=False),
         intervalo_h=widgets.FloatSlider(value=24, min=2, max=72, step=1, description='Cada (h)',
                                         continuous_update=False),
         dias=widgets.IntSlider(value=7, min=1, max=14, description='Días', continuous_update=False));

### **Problema 5**
Una lámina de 1 µm de espesor y 1 cm² de sección de $^{197}$Au ($A$ = 196,97, $\rho$ = 19,3 g cm⁻³) es irradiada con un flujo de $2,2\times10^{12}$ cm⁻² s⁻¹ de neutrones térmicos cuya sección eficaz de captura es 98,8 barn. El isótopo $^{198}$Au producido tiene un periodo de 2,7 días.

**a)** Hallar la actividad de la lámina al cabo de 10 días de irradiación.

**b)** El $^{198}$Au se desintegra básicamente por emisión $\beta^-$ quedando en un estado excitado de $^{198}$Hg que se desexcita al fundamental por emisión de un fotón de 412 keV. Los coeficientes de conversión interna de esta transición son: $\alpha_K$ = 0,0323 y $\alpha_L$ = 0,0149. Las energías de enlace de las capas K y L son $E_K$ = 20 keV y $E_L$ = 9 keV.
* Hallar el número de electrones y de fotones emitidos por segundo por la muestra a los 10 días de irradiación.
* Dibujar esquemáticamente el espectro de energía de los fotones y electrones emitidos por la muestra.

---

#### **a) Activación neutrónica**
Sea A = $^{197}$Au (blanco) y B = $^{198}$Au (producto). La tasa de producción por unidad de volumen es $N_A\sigma\phi$, de modo que:
$$ \frac{dN_A}{dt} = -\sigma\phi N_A, \qquad \frac{dN_B}{dt} = \sigma\phi N_A - \lambda N_B $$

**Comprobaciones previas:**
* Densidad de núcleos: $N_A = \frac{N_a}{P_a}\rho = \frac{6.02\times10^{23}}{196.97}\times19.3 = 5.90\times10^{22}$ cm⁻³.
* El haz **no se atenúa**: $N_A\sigma d = 5.90\times10^{22}\times98.8\times10^{-24}\times10^{-4} = 5.8\times10^{-4} \ll 1$.
* El blanco **no se agota**: $\sigma\phi = 2.2\times10^{-10}$ s⁻¹ $\ll\lambda = 2.97\times10^{-6}$ s⁻¹ (en 10 días sólo se transforma una fracción $\sigma\phi t \approx 2\times10^{-4}$).

Por tanto $N_A$ es constante y:
$$ A_V(t) = \lambda N_B = N_A\sigma\phi\left(1 - e^{-\lambda t}\right) $$
La actividad tiende a la **saturación** $N_A\sigma\phi = 1.28\times10^{13}$ s⁻¹cm⁻³. A los 10 días ($e^{-\lambda t} = 0.077$):
$$ A_V = 1.18\times10^{13}\text{ s}^{-1}\text{cm}^{-3} \quad\Rightarrow\quad A_\beta = A_V\times S\times d = 1.18\times10^{13}\times10^{-4} = \mathbf{1.18\times10^9\text{ s}^{-1}} \;(32\text{ mCi}) $$

#### **b) Electrones y fotones emitidos**
El $^{198}$Hg excitado se desexcita inmediatamente (equilibrio secular), de modo que cada desintegración $\beta$ va seguida de un $\gamma$ **o** de un electrón de conversión. Con $\alpha = \alpha_K + \alpha_L = 0.0472$:
$$ A_\gamma = \frac{A_\beta}{1+\alpha} = \mathbf{1.13\times10^9\text{ fotones/s}}, \qquad A_c = A_\beta\frac{\alpha}{1+\alpha} = 5.3\times10^7\text{ e}^-\text{/s} $$
Los electrones son los $\beta$ más los de conversión:
$$ A_e = A_\beta\left(1 + \frac{\alpha}{1+\alpha}\right) = \mathbf{1.24\times10^9\text{ electrones/s}} $$

**Espectro:**
* **Fotones:** una línea a **412 keV** (más rayos X característicos del Hg tras las conversiones).
* **Electrones:** el **espectro continuo $\beta$** (hasta $E_{max} \approx 961$ keV) y superpuestos dos picos de conversión: **K a 392 keV** y **L a 403 keV**, con un cociente $I_K/I_L = \alpha_K/\alpha_L = 2.17$. El área total de los picos de conversión es un 4.5 % del área del espectro $\beta$.

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
rho_Au, Pa_Au = 19.3, 196.97
d_Au, S_Au = 1e-4, 1.0          # cm, cm²
phi_n = 2.2e12                   # cm^-2 s^-1
sigma_Au = 98.8 * barn
T_Au198 = 2.7 * dia
aK5, aL5 = 0.0323, 0.0149
E_g5, EK5, EL5 = 412.0, 20.0, 9.0

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
N_Au = NA / Pa_Au * rho_Au
l_Au = lam(T_Au198)
sat_V = N_Au * sigma_Au * phi_n
A_V10 = sat_V * (1 - np.exp(-l_Au * 10 * dia))
A_beta5 = A_V10 * S_Au * d_Au
a5 = aK5 + aL5
A_gamma5 = A_beta5 / (1 + a5)
A_conv5 = A_beta5 * a5 / (1 + a5)

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 5)                    ")
print("=====================================================================")
print(f"N(197Au) = {N_Au:.3e} cm⁻³,   λ = {l_Au:.3e} s⁻¹")
print(f"Atenuación del haz: Nσd = {N_Au*sigma_Au*d_Au:.1e}   |   Agotamiento: σφ·10 d = {sigma_Au*phi_n*10*dia:.1e}")
print(f"a) Saturación: {sat_V:.3e} s⁻¹cm⁻³  |  A_V(10 d) = {A_V10:.3e} s⁻¹cm⁻³")
print(f"   A_β(10 d) = {A_beta5:.3e} s⁻¹ = {A_beta5/Ci*1e3:.1f} mCi")
print("-" * 69)
print(f"b) Fotones (412 keV): {A_gamma5:.3e} s⁻¹")
print(f"   Electrones de conversión: {A_conv5:.3e} s⁻¹  (K: {A_beta5*aK5/(1+a5):.2e} a {E_g5-EK5:.0f} keV;"
      f" L: {A_beta5*aL5/(1+a5):.2e} a {E_g5-EL5:.0f} keV)")
print(f"   Electrones totales (β + conversión): {A_beta5 + A_conv5:.3e} s⁻¹")
print("=====================================================================")

# Espectro esquemático (forma β aproximada con la función de Fermi despreciada)
E_max = 961.0
E = np.linspace(0, 1000, 2000)
me_keV = 511.0
W = E + me_keV; p = np.sqrt(np.maximum(W**2 - me_keV**2, 0))
beta = np.where(E < E_max, p * W * (E_max - E)**2, 0); beta /= beta.sum() * (E[1] - E[0])
def pico(E0, area, s=3.0):
    return area * np.exp(-0.5 * ((E - E0) / s)**2) / (np.sqrt(2 * np.pi) * s)
electrones = A_beta5 * beta + pico(E_g5 - EK5, A_beta5 * aK5 / (1 + a5)) + pico(E_g5 - EL5, A_beta5 * aL5 / (1 + a5))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].fill_between(E, pico(E_g5, A_gamma5), color='#4c72b0', alpha=0.6)
axes[0].text(E_g5 + 10, pico(E_g5, A_gamma5).max() * 0.8, '412 keV', color='#4c72b0')
axes[0].set_ylabel('Fotones (s⁻¹ keV⁻¹)'); axes[0].set_title('Espectro esquemático de la lámina de ¹⁹⁸Au', fontweight='bold')
axes[1].fill_between(E, electrones, color='#c44e52', alpha=0.6)
axes[1].text(E_g5 - EK5 - 120, electrones.max() * 0.85, 'Pico K (392 keV)', color='#c44e52')
axes[1].text(E_g5 - EL5 + 15, electrones.max() * 0.45, 'Pico L (403 keV)', color='#c44e52')
axes[1].set_xlabel('Energía (keV)'); axes[1].set_ylabel('Electrones (s⁻¹ keV⁻¹)')
plt.tight_layout(); plt.show()

#### **Bonus: Simulador de activación neutrónica (Problemas 5, 6, 7 y 8)**

El widget simula una irradiación de duración $t_{irr}$ seguida de un periodo de enfriamiento. Por defecto usa los datos del Problema 5 (lámina de oro). Puedes activar el **agotamiento del blanco** (para flujos enormes, $\sigma\phi$ comparable a $\lambda$), en cuyo caso la solución exacta es:
$$ A_B(t) = N_A(0)\,\sigma\phi\,\frac{\lambda}{\lambda - \sigma\phi}\left(e^{-\sigma\phi t} - e^{-\lambda t}\right) $$
(es decir, la misma ecuación de Bateman con "$\lambda_A$" $= \sigma\phi$).

**Cuestiones:**
1. ¿Qué fracción de la actividad de saturación se alcanza tras 1, 2 y 5 periodos de irradiación? ¿Merece la pena irradiar más de ~3 periodos?
2. Multiplica el flujo por $10^4$. ¿Sigue siendo válida la hipótesis de blanco no agotado?
3. Tras cesar la irradiación, ¿con qué periodo decae la actividad?

In [ ]:
def activacion(log_flujo, sigma_b, T_dias, t_irr_dias, agotamiento):
    phi = 10**log_flujo
    sp = sigma_b * barn * phi
    l = lam(T_dias * dia)
    N0 = N_Au * S_Au * d_Au                       # núcleos blanco en la lámina
    t_irr = t_irr_dias * dia
    t = np.linspace(0, t_irr + 3 * T_dias * dia, 1500)
    def A_durante(tt):
        if agotamiento:
            return bateman_hija(N0 * sp, sp, l, tt)
        return N0 * sp * (1 - np.exp(-l * tt))
    A_fin = A_durante(t_irr)
    A = np.where(t <= t_irr, A_durante(np.minimum(t, t_irr)), A_fin * np.exp(-l * (t - t_irr)))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(t / dia, A / Ci * 1e3, color='#c44e52', lw=2.5)
    ax.axhline(N0 * sp / Ci * 1e3, color='gray', ls='--', label='Saturación N σ φ (blanco no agotado)')
    ax.axvspan(0, t_irr_dias, color='gold', alpha=0.2, label='Irradiación')
    ax.set_xlabel('Tiempo (días)'); ax.set_ylabel('Actividad (mCi)')
    ax.set_title(f'Activación: φ = {phi:.1e} cm⁻²s⁻¹, σ = {sigma_b:.1f} b, T = {T_dias:.2f} d', fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"Actividad al final de la irradiación: {A_fin:.3e} s⁻¹ = {A_fin/Ci*1e3:.2f} mCi "
          f"({100*A_fin/(N0*sp):.1f}% de la saturación)   |   σφ/λ = {sp/l:.2e}")

interact(activacion,
         log_flujo=widgets.FloatSlider(value=np.log10(2.2e12), min=8, max=18, step=0.1, description='log₁₀ φ',
                                       continuous_update=False),
         sigma_b=widgets.FloatSlider(value=98.8, min=1, max=1000, step=0.1, description='σ (barn)',
                                     continuous_update=False),
         T_dias=widgets.FloatSlider(value=2.7, min=0.1, max=30, step=0.1, description='T (días)',
                                    continuous_update=False),
         t_irr_dias=widgets.FloatSlider(value=10, min=0.5, max=60, step=0.5, description='t_irr (días)',
                                        continuous_update=False),
         agotamiento=widgets.Checkbox(value=False, description='Incluir agotamiento del blanco'));

### **Problema 6**
Una lámina muy fina de 1 mg de plata es irradiada por un haz de $1,00\times10^{10}$ cm⁻² s⁻¹ neutrones lentos que, por interacción con los isótopos estables $^{107}$Ag (51,8 %) y $^{109}$Ag (48,2 %), producen los isótopos radiactivos beta $^{108}$Ag y $^{110}$Ag. Empleando los datos adjuntos resolver las siguientes cuestiones:

**a)** Estimar el tiempo de irradiación necesario para alcanzar la saturación. Una vez alcanzada ésta, se apaga el haz de neutrones ($t = 0$). Calcular la actividad beta de la muestra en función del tiempo $A(t)$.

**b)** ¿Qué tiempo tiene que transcurrir desde que cesa el haz de neutrones para que las actividades de ambos isótopos sean iguales? ¿Permanecen iguales con el tiempo? Dibujar en una sola gráfica $\log A$ frente al tiempo $t$ para cada isótopo y su valor total.

**DATOS:** $P_a$ = 107,87 g, $N_a = 6,02\times10^{23}$, $\sigma(^{107}\text{Ag})$ = 31 barn, $\sigma(^{109}\text{Ag})$ = 87 barn, $T(^{108}\text{Ag})$ = 2,37 min, $T(^{110}\text{Ag})$ = 24,6 s.

---

#### **a) Saturación y actividad tras la irradiación**
Número de núcleos de cada isótopo:
$$ N = \frac{N_a}{P_a}m = \frac{6.02\times10^{23}}{107.87}\times10^{-3} = 5.58\times10^{18} \;\Rightarrow\; N_{107} = 2.89\times10^{18},\quad N_{109} = 2.69\times10^{18} $$
Constantes de desintegración: $\lambda_{108} = \ln2/142.2\text{ s} = 4.87\times10^{-3}$ s⁻¹; $\lambda_{110} = \ln2/24.6\text{ s} = 2.82\times10^{-2}$ s⁻¹. Como $\sigma\phi \sim 10^{-13}$ s⁻¹ $\ll \lambda$, el blanco no se agota y (lámina muy fina) el haz no se atenúa:
$$ A_{108}(t) = N_{107}\sigma_{107}\phi\left(1 - e^{-\lambda_{108}t}\right), \qquad A_{110}(t) = N_{109}\sigma_{109}\phi\left(1 - e^{-\lambda_{110}t}\right) $$
Actividades de saturación: $A_{108}^{sat} = 8.96\times10^5$ s⁻¹, $A_{110}^{sat} = 2.34\times10^6$ s⁻¹, total $3.24\times10^6$ s⁻¹.

La saturación la marca el isótopo de **periodo más largo** ($^{108}$Ag): la actividad total alcanza el 90 % de la saturación a los **213 s** y el 98 % a los **539 s** ($\approx 4$ periodos del $^{108}$Ag). Unos **10 minutos** de irradiación son suficientes.

Tras apagar el haz ($t = 0$):
$$ A(t) = A_{108}^{sat}e^{-\lambda_{108}t} + A_{110}^{sat}e^{-\lambda_{110}t} = 8.96\times10^5 e^{-t/205\text{ s}} + 2.34\times10^6 e^{-t/35.5\text{ s}}\ \text{s}^{-1} $$

#### **b) Igualdad de actividades**
$$ A_{108}^{sat}e^{-\lambda_{108}t} = A_{110}^{sat}e^{-\lambda_{110}t} \quad\Longrightarrow\quad t = \frac{\ln\left(A_{110}^{sat}/A_{108}^{sat}\right)}{\lambda_{110} - \lambda_{108}} = \frac{\ln 2.61}{2.33\times10^{-2}\text{ s}^{-1}} = \mathbf{41\text{ s}} $$
Las actividades **no permanecen iguales**: el $^{110}$Ag, de periodo más corto, decae más deprisa, y a partir de entonces domina el $^{108}$Ag. En la gráfica de $\log A$, cada isótopo es una recta y la total es una curva que, a tiempos largos, se confunde con la recta del $^{108}$Ag. Esta es la técnica clásica para **separar componentes** en una curva de desintegración compuesta: se ajusta la cola a una recta, se resta, y se obtiene la componente rápida.

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
m_Ag, Pa_Ag = 1e-3, 107.87
phi_Ag = 1.0e10
ab107, ab109 = 0.518, 0.482
s107, s109 = 31 * barn, 87 * barn
T108, T110 = 2.37 * 60, 24.6          # s

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
N_Ag = NA / Pa_Ag * m_Ag
N107, N109 = ab107 * N_Ag, ab109 * N_Ag
l108, l110 = lam(T108), lam(T110)
A108s, A110s = N107 * s107 * phi_Ag, N109 * s109 * phi_Ag
A_sat = A108s + A110s
frac = lambda t: (A108s * (1 - np.exp(-l108 * t)) + A110s * (1 - np.exp(-l110 * t))) / A_sat
t90 = brentq(lambda t: frac(t) - 0.90, 1, 1e4)
t98 = brentq(lambda t: frac(t) - 0.98, 1, 1e4)
t_igual_Ag = np.log(A110s / A108s) / (l110 - l108)

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 6)                    ")
print("=====================================================================")
print(f"N(107Ag) = {N107:.3e},  N(109Ag) = {N109:.3e}")
print(f"λ(108Ag) = {l108:.3e} s⁻¹,  λ(110Ag) = {l110:.3e} s⁻¹,  σφ ~ {s109*phi_Ag:.1e} s⁻¹")
print(f"a) Saturación: A108 = {A108s:.3e} s⁻¹,  A110 = {A110s:.3e} s⁻¹,  total = {A_sat:.3e} s⁻¹")
print(f"   90% de la saturación a los {t90:.0f} s;  98% a los {t98:.0f} s")
print(f"b) Actividades iguales a los {t_igual_Ag:.1f} s tras cesar la irradiación")
print("=====================================================================")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
t = np.linspace(0, 1400, 600)
ax = axes[0]
ax.plot(t, A108s * (1 - np.exp(-l108 * t)), color='#4c72b0', lw=2, label='¹⁰⁸Ag')
ax.plot(t, A110s * (1 - np.exp(-l110 * t)), color='#c44e52', lw=2, label='¹¹⁰Ag')
ax.plot(t, A_sat * frac(t), color='#55a868', lw=2.5, label='Total')
for tx, p in [(t90, 90), (t98, 98)]:
    ax.axvline(tx, color='gray', ls=':'); ax.text(tx, 0.3e6, f' {p}%\n {tx:.0f} s', color='gray')
ax.set_xlabel('Tiempo de irradiación (s)'); ax.set_ylabel('Actividad (s⁻¹)')
ax.set_title('a) Activación de la plata', fontweight='bold'); ax.legend()
ax = axes[1]
t = np.linspace(0, 300, 600)
ax.semilogy(t, A108s * np.exp(-l108 * t), color='#4c72b0', lw=2, label='¹⁰⁸Ag (T = 2.37 min)')
ax.semilogy(t, A110s * np.exp(-l110 * t), color='#c44e52', lw=2, label='¹¹⁰Ag (T = 24.6 s)')
ax.semilogy(t, A108s * np.exp(-l108 * t) + A110s * np.exp(-l110 * t), color='#55a868', lw=2.5, label='Total')
ax.axvline(t_igual_Ag, color='gray', ls='--'); ax.text(t_igual_Ag + 3, 2e6, f'{t_igual_Ag:.0f} s', color='gray')
ax.set_ylim(1e4, 5e6)
ax.set_xlabel('Tiempo desde el fin de la irradiación (s)'); ax.set_ylabel('Actividad (s⁻¹)')
ax.set_title('b) Desintegración: log A frente a t', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.show()

### **Problema 7**
Un haz de $1,00\cdot10^6$ neutrones cm⁻² s⁻¹ atraviesa una lámina de 1,0 cm de espesor de tántalo ($^{181}$Ta). El haz emergente tiene $5,82\cdot10^5$ neutrones cm⁻² s⁻¹. La actividad de la lámina después de 60 días de irradiación y medida 30 días después del cese de ésta es de $2,23\cdot10^4$ desintegraciones cm⁻² s⁻¹ debidas al isótopo $^{182}$Ta. Resolver las siguientes cuestiones:

**a)** Calcular la sección eficaz total de interacción de neutrones con $^{181}$Ta.

**b)** Calcular la sección eficaz para la producción de $^{182}$Ta despreciando el efecto de la atenuación del haz.

**c)** Repetir el apartado b) teniendo en cuenta el efecto de la atenuación del haz.

**DATOS:** La densidad del tántalo es de 16,7 g cm⁻³ y el periodo del $^{182}$Ta es $T$ = 114 días.

---

#### **a) Sección eficaz total**
$N = \frac{N_a}{P_a}\rho = \frac{6.02\times10^{23}}{181}\times16.7 = 5.56\times10^{22}$ cm⁻³. El haz se atenúa exponencialmente por **todas** las interacciones (captura, scattering...):
$$ \phi_1 = \phi_0\,e^{-N\sigma_T d} \quad\Longrightarrow\quad \sigma_T = \frac{1}{Nd}\ln\frac{\phi_0}{\phi_1} = \frac{\ln(1/0.582)}{5.56\times10^{22}\times1.0} = \mathbf{9.74\times10^{-24}\text{ cm}^2} \;(9.7\text{ b}) $$

#### **b) Sección eficaz de producción de $^{182}$Ta (sin atenuación)**
La actividad por cm² de lámina tras $t_1 = 60$ días de irradiación y $t_2 = 30$ días de enfriamiento ($\lambda = \ln2/114\text{ d} = 6.08\times10^{-3}$ d⁻¹) es:
$$ A = N d\,\sigma_a\,\phi_0\left(1 - e^{-\lambda t_1}\right)e^{-\lambda t_2} $$
$$ \sigma_a = \frac{A}{Nd\,\phi_0\left(1 - e^{-\lambda t_1}\right)e^{-\lambda t_2}} = \frac{2.23\times10^4}{5.56\times10^{22}\times10^6\times0.306\times0.833} = \mathbf{1.58\times10^{-24}\text{ cm}^2} $$

#### **c) Con atenuación del haz**
La lámina es **gruesa** ($N\sigma_T d = 0.54$): el flujo disminuye con la profundidad, $\phi(x) = \phi_0 e^{-N\sigma_T x}$, y las capas profundas se activan menos. Integrando la producción a lo largo del espesor:
$$ A = N\sigma_a\phi_0\left(1 - e^{-\lambda t_1}\right)e^{-\lambda t_2}\int_0^d e^{-N\sigma_T x}dx = \frac{\sigma_a}{\sigma_T}\,\phi_0\left(1 - e^{-N\sigma_T d}\right)\left(1 - e^{-\lambda t_1}\right)e^{-\lambda t_2} $$
El factor $\left(1 - e^{-N\sigma_T d}\right)$ es la probabilidad de que un neutrón interaccione en la lámina, y $\sigma_a/\sigma_T$ la fracción de interacciones que son capturas. Despejando:
$$ \sigma_a = \frac{A\,\sigma_T}{\phi_0\left(1 - e^{-N\sigma_T d}\right)\left(1 - e^{-\lambda t_1}\right)e^{-\lambda t_2}} = \mathbf{2.04\times10^{-24}\text{ cm}^2} $$
Despreciar la atenuación subestima la sección eficaz en un 23 %. (Comprobación: $\sigma_a = 2.04$ b $< \sigma_T = 9.7$ b, como debe ser.)

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
phi0_Ta, phi1_Ta = 1.00e6, 5.82e5
d_Ta, rho_Ta, Pa_Ta = 1.0, 16.7, 181.0
A_Ta = 2.23e4                    # cm^-2 s^-1
t1, t2 = 60.0, 30.0              # días
T_Ta182 = 114.0                  # días

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
N_Ta = NA / Pa_Ta * rho_Ta
sT = np.log(phi0_Ta / phi1_Ta) / (N_Ta * d_Ta)
lT = lam(T_Ta182)
f_tiempo = (1 - np.exp(-lT * t1)) * np.exp(-lT * t2)
sa_fina = A_Ta / (N_Ta * d_Ta * phi0_Ta * f_tiempo)
sa_gruesa = A_Ta * sT / (phi0_Ta * (1 - np.exp(-N_Ta * sT * d_Ta)) * f_tiempo)

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 7)                    ")
print("=====================================================================")
print(f"N(181Ta) = {N_Ta:.3e} cm⁻³,  λ = {lT:.3e} d⁻¹")
print(f"a) σ_T = {sT:.3e} cm² = {sT/barn:.2f} b    (N σ_T d = {N_Ta*sT*d_Ta:.3f})")
print(f"b) σ_a (lámina fina)   = {sa_fina:.3e} cm² = {sa_fina/barn:.2f} b")
print(f"c) σ_a (con atenuación) = {sa_gruesa:.3e} cm² = {sa_gruesa/barn:.2f} b")
print(f"   Factor de corrección por autoapantallamiento: {sa_gruesa/sa_fina:.3f}")
print("=====================================================================")

x = np.linspace(0, d_Ta, 200)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, np.exp(-N_Ta * sT * x), color='#4c72b0', lw=2.5, label='φ(x)/φ₀ (flujo real)')
ax.axhline(1, color='#c44e52', ls='--', label='Hipótesis de lámina fina')
ax.fill_between(x, np.exp(-N_Ta * sT * x), 1, color='#c44e52', alpha=0.15, label='Producción "perdida" por atenuación')
ax.set_xlabel('Profundidad en el tántalo (cm)'); ax.set_ylabel('φ(x)/φ₀'); ax.set_ylim(0, 1.1)
ax.set_title('Autoapantallamiento: el flujo disminuye dentro de la lámina', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

### **Problema 8**
Un haz de neutrones lentos de 1,0 cm² de sección incide sobre una lámina de 1 µm de espesor de $^{197}$Au (densidad $\rho$ = 19,3 g cm⁻³). La absorción del neutrón produce $^{198}$Au, siendo la sección eficaz del proceso $\sigma$ = 98,8 barn. No existe ningún otro proceso de interacción relevante. El $^{198}$Au se desintegra por emisión $\beta^-$ convirtiéndose en el núcleo estable $^{198}$Hg.

**a)** ¿Cuántos neutrones por segundo debe de transportar el haz para poder conseguir una muestra radiactiva de $^{198}$Au de 500 mCi al alcanzar la saturación? 1 Ci = $3,7\times10^{10}$ s⁻¹.

**b)** Empleando el concepto de sección eficaz y suponiendo que la desintegración del $^{198}$Au es instantánea, hallar el tiempo necesario para convertir un 1 % del oro en mercurio con este haz de neutrones.

---

#### **a) Intensidad del haz**
Como en el Problema 5, $N = 5.90\times10^{22}$ cm⁻³ y el haz no se atenúa ($N\sigma d = 5.8\times10^{-4}$). La actividad de saturación es:
$$ A_{sat} = N\,S\,d\,\sigma\,\phi = 5.82\times10^{-4}\text{ cm}^2\times\phi = 500\text{ mCi} = 1.85\times10^{10}\text{ s}^{-1} $$
$$ \phi = \frac{1.85\times10^{10}}{5.82\times10^{-4}} = 3.18\times10^{13}\text{ cm}^{-2}\text{s}^{-1} \quad\Rightarrow\quad \mathbf{3.18\times10^{13}\text{ neutrones/s}} \text{ (haz de 1 cm²)} $$
Es un flujo típico del interior de un reactor de investigación.

#### **b) Transmutación del 1 % del oro**
La probabilidad por unidad de tiempo de que un núcleo de Au capture un neutrón es $\sigma\phi$. Para convertir una fracción pequeña $f = 0.01$:
$$ \sigma\phi\,t = 0.01 \quad\Rightarrow\quad t = \frac{0.01}{\sigma\phi} = \frac{0.01}{98.8\times10^{-24}\times3.18\times10^{13}} = 3.19\times10^6\text{ s} = \mathbf{36.9\text{ días}} $$
(El cálculo exacto, teniendo en cuenta que el oro se va agotando, $1 - e^{-\sigma\phi t} = 0.01$, da 37.1 días.)

> Ésta es la "piedra filosofal" de los alquimistas... al revés: ¡convertir oro en mercurio!

In [ ]:
# =====================================================================
# 1. DATOS Y CÁLCULOS
# =====================================================================
A_obj = 0.5 * Ci
NSd = N_Au * S_Au * d_Au * sigma_Au     # cm² (sección eficaz macroscópica total de la lámina)
phi_8 = A_obj / NSd
t_1pc = 0.01 / (sigma_Au * phi_8)
t_1pc_exacto = -np.log(0.99) / (sigma_Au * phi_8)

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 8)                    ")
print("=====================================================================")
print(f"N S d σ = {NSd:.3e} cm²   (N σ d = {N_Au*d_Au*sigma_Au:.1e}: el haz no se atenúa)")
print(f"a) φ = {phi_8:.3e} cm⁻² s⁻¹  ->  {phi_8*S_Au:.3e} neutrones/s")
print(f"b) t(1%) = {t_1pc:.3e} s = {t_1pc/dia:.1f} días   (exacto: {t_1pc_exacto/dia:.1f} días)")
print("=====================================================================")

### **Problema 9**
Se ha medido la actividad $\beta$ de cierta muestra que contiene 1,20 g de carbono. En un intervalo de 24 h se han detectado un total de 36288 desintegraciones. Una medida independiente del fondo ha dado 17424 cuentas en 24 h. Sabiendo que el porcentaje de $^{14}$C en un vegetal vivo es de $1,35\cdot10^{-10}$ % del total de carbono y que el periodo del $^{14}$C es 5730 años, hallar la antigüedad de la muestra y el error en esta determinación. El peso atómico del carbono es 12,01.

---

#### **1. Actividad específica del carbono vivo**
$$ N_C = \frac{N_a}{P_a} = 5.01\times10^{22}\text{ g}^{-1}, \qquad \lambda = \frac{\ln 2}{5730\text{ años}} = 2.30\times10^{-10}\text{ min}^{-1}, \qquad f_v = 1.35\times10^{-12} $$
$$ A_v = \lambda\,f_v\,N_C = \mathbf{15.6\text{ desintegraciones g}^{-1}\text{ min}^{-1}} $$

#### **2. Actividad específica de la muestra**
Cuentas netas en 24 h: $D = 36288 - 17424 = 18864$.
$$ A_m = \frac{18864}{1440\text{ min}\times1.20\text{ g}} = 10.9\text{ g}^{-1}\text{min}^{-1} $$

#### **3. Antigüedad**
Tras la muerte, el $^{14}$C deja de renovarse y su actividad decae: $A_m = A_v e^{-\lambda t}$.
$$ t = \frac{1}{\lambda}\ln\frac{A_v}{A_m} = \frac{1}{2.30\times10^{-10}\text{ min}^{-1}}\ln\frac{15.6}{10.9} \approx \mathbf{2930\text{ años}} $$

#### **4. Incertidumbre**
Total y fondo son variables de Poisson independientes:
$$ \sigma(D) = \sqrt{T + F} = \sqrt{36288 + 17424} = 232 \quad\Rightarrow\quad \frac{\Delta A_m}{A_m} = \frac{232}{18864} = 1.2\% $$
Despreciando las incertidumbres en $\lambda$ y $A_v$:
$$ \Delta t = \frac{1}{\lambda}\frac{\Delta A_m}{A_m} = 8267\text{ años}\times0.0123 \approx 100\text{ años} \quad\Rightarrow\quad \boxed{t = (2930 \pm 100)\text{ años}} $$

> La incertidumbre relativa $\frac{\Delta D}{D} = \frac{1}{\sqrt{T}}\frac{\sqrt{1 + F/T}}{1 - F/T}$ crece mucho cuando el fondo es comparable a la señal: en muestras antiguas (poca señal) el fondo es el enemigo. Por eso la datación por $^{14}$C con contadores está limitada a unos 40000-50000 años (la espectrometría de masas con aceleradores, AMS, cuenta directamente los átomos de $^{14}$C y llega más lejos).

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
m_C = 1.20               # g
T_cuentas, F_cuentas = 36288, 17424
t_med = 24 * 60.0        # min
f_viva = 1.35e-12
T_C14 = 5730.0           # años
Pa_C = 12.01

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
l_C14_min = lam(T_C14 * 365.25 * 24 * 60)           # min^-1
A_viva = l_C14_min * f_viva * NA / Pa_C              # g^-1 min^-1
D = T_cuentas - F_cuentas
sD = np.sqrt(T_cuentas + F_cuentas)
A_muestra = D / (t_med * m_C)
edad = np.log(A_viva / A_muestra) / lam(T_C14)
d_edad = (sD / D) / lam(T_C14)

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 9)                    ")
print("=====================================================================")
print(f"λ = {l_C14_min:.3e} min⁻¹,  A_viva = {A_viva:.2f} g⁻¹ min⁻¹")
print(f"Cuentas netas D = {D} ± {sD:.0f}  ({100*sD/D:.2f}%)")
print(f"A_muestra = {A_muestra:.2f} g⁻¹ min⁻¹")
print(f"Antigüedad: t = ({edad:.0f} ± {d_edad:.0f}) años")
print("=====================================================================")

#### **Bonus: Simulador de datación por radiocarbono**

Elige la edad verdadera de una muestra, su masa de carbono, la tasa de fondo del contador y el tiempo de medida. El widget simula la medida (cuentas poissonianas de señal + fondo y una medida de fondo de igual duración) y estima la edad con su incertidumbre.

**Cuestiones:**
1. Con los parámetros del problema (1.2 g, fondo de 12 cpm, 24 h), ¿a partir de qué edad la incertidumbre supera los 1000 años?
2. ¿Cuánto hay que medir para datar con ±100 años una muestra de 10000 años?
3. ¿Por qué es tan importante reducir el fondo del contador (blindajes, anticoincidencias)?

In [ ]:
rng = np.random.default_rng(14)

def datacion_C14(edad_real, masa_g, fondo_cpm, horas):
    t_min = horas * 60
    lC = lam(T_C14)
    tasa_senal = A_viva * masa_g * np.exp(-lC * edad_real)        # cpm (eficiencia 100%)
    T = rng.poisson((tasa_senal + fondo_cpm) * t_min)
    F = rng.poisson(fondo_cpm * t_min)
    D = T - F
    if D <= 0:
        print(f"⚠️ Cuentas netas ≤ 0 (T = {T}, F = {F}): la muestra es indistinguible del fondo.")
        return
    edad_est = np.log(A_viva * masa_g * t_min / D) / lC
    s_edad = np.sqrt(T + F) / D / lC
    # Incertidumbre esperada en función de la edad
    edades = np.linspace(0, 60000, 400)
    S = A_viva * masa_g * np.exp(-lC * edades) * t_min
    B = fondo_cpm * t_min
    s_esp = np.sqrt(S + 2 * B) / S / lC
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.semilogy(edades, s_esp, color='#4c72b0', lw=2.5, label='Incertidumbre esperada')
    ax.errorbar(edad_real, s_edad, fmt='*', color='gold', ms=18, mec='k', label=f'Esta medida: {edad_est:.0f} ± {s_edad:.0f} años')
    ax.axhline(100, color='gray', ls=':'); ax.axhline(1000, color='gray', ls=':')
    ax.set_ylim(10, 1e5); ax.set_xlabel('Edad de la muestra (años)'); ax.set_ylabel('σ(edad) (años)')
    ax.set_title(f'{masa_g:.1f} g de C, fondo {fondo_cpm:.1f} cpm, {horas:.0f} h de medida', fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"Señal esperada: {tasa_senal:.2f} cpm  |  T = {T}, F = {F}  |  Edad estimada: {edad_est:.0f} ± {s_edad:.0f} años "
          f"(verdadera {edad_real:.0f})")

interact(datacion_C14,
         edad_real=widgets.FloatSlider(value=2930, min=0, max=50000, step=10, description='Edad (años)',
                                       continuous_update=False),
         masa_g=widgets.FloatSlider(value=1.2, min=0.1, max=10, step=0.1, description='Masa C (g)',
                                    continuous_update=False),
         fondo_cpm=widgets.FloatSlider(value=17424 / 1440, min=0.1, max=30, step=0.1, description='Fondo (cpm)',
                                       continuous_update=False),
         horas=widgets.FloatSlider(value=24, min=1, max=500, step=1, description='Horas',
                                   continuous_update=False));

### **Problema 10**
El cuerpo humano contiene en media alrededor del 18 % de carbono y 0,2 % de potasio. El $^{14}$C tiene un periodo de 5730 años, se desintegra por emisión $\beta^-$ al estado fundamental del $^{14}$N y su proporción en el carbono de la materia viva es de $1,35\times10^{-10}$ %. El $^{40}$K tiene un periodo de $1,28\times10^9$ años y se encuentra en una proporción del 0,012 %. El $^{40}$K se desintegra por emisión $\beta^-$ (89,2 %), convirtiéndose en el estado fundamental del $^{40}$Ca, o por captura electrónica (10,7 %), pasando a un estado excitado del $^{40}$Ar que decae al fundamental por emisión de un rayo $\gamma$. Los pesos atómicos del potasio y del carbono son 39,1 y 12,01 respectivamente. Hallar la actividad intrínseca media de una persona de 70 kg debido a la desintegración del $^{14}$C y del $^{40}$K.

---

#### **1. Actividad específica del carbono vivo**
$$ A(^{14}\text{C}) = \lambda f N = \frac{\ln 2}{5730\text{ años}}\times1.35\times10^{-12}\times\frac{6.02\times10^{23}}{12.01} = 0.259\text{ g}^{-1}\text{s}^{-1} \text{ (por g de carbono)} $$
Por gramo de persona: $0.259\times0.18 = 0.047$ g⁻¹s⁻¹.

#### **2. Actividad específica del potasio natural**
$$ A(^{40}\text{K}) = \lambda f N = \frac{\ln 2}{1.28\times10^9\text{ años}}\times1.2\times10^{-4}\times\frac{6.02\times10^{23}}{39.1} = 31.7\text{ g}^{-1}\text{s}^{-1} \text{ (por g de K)} $$
Por gramo de persona: $31.7\times0.002 = 0.063$ g⁻¹s⁻¹. (La actividad incluye ambas ramas, $\beta^-$ y captura electrónica.)

#### **3. Persona de 70 kg**
$$ A_C = 0.0467\times70000 \approx 3270\text{ s}^{-1}, \qquad A_K = 0.063\times70000 \approx 4440\text{ s}^{-1} $$
$$ A_{total} \approx 7700\text{ Bq} \approx \mathbf{0.2\ \mu\text{Ci}} $$
¡Unas 8000 desintegraciones por segundo en nuestro propio cuerpo! El $^{40}$K es además la principal fuente de dosis interna natural, porque su $\beta$ es mucho más energética (1.31 MeV de energía máxima frente a 0.156 MeV del $^{14}$C) y emite un $\gamma$ de 1.46 MeV.

> Las soluciones de clase usan una abundancia de $^{40}$K de 0.0119 %, con lo que obtienen 31.5 g⁻¹s⁻¹ y $A_K = 4390$ s⁻¹; la diferencia con el 0.012 % del enunciado es inferior al 1 %.

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
masa_persona = 70e3       # g
frac_C, frac_K = 0.18, 0.002
f_K40 = 0.012e-2
T_K40 = 1.28e9            # años
Pa_K = 39.1

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
A_C_g = lam(T_C14 * anio) * f_viva * NA / Pa_C        # s^-1 por g de C
A_K_g = lam(T_K40 * anio) * f_K40 * NA / Pa_K         # s^-1 por g de K
A_C_persona = A_C_g * frac_C * masa_persona
A_K_persona = A_K_g * frac_K * masa_persona
A_tot = A_C_persona + A_K_persona

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 10)                   ")
print("=====================================================================")
print(f"A(14C) = {A_C_g:.3f} s⁻¹ por g de C   ->  {A_C_g*frac_C:.4f} s⁻¹ por g de persona")
print(f"A(40K) = {A_K_g:.1f} s⁻¹ por g de K    ->  {A_K_g*frac_K:.4f} s⁻¹ por g de persona")
print(f"Persona de 70 kg:  A_C = {A_C_persona:.0f} Bq,  A_K = {A_K_persona:.0f} Bq  (β: {0.892*A_K_persona:.0f}, CE: {0.107*A_K_persona:.0f})")
print(f"Total: {A_tot:.0f} Bq = {A_tot/Ci*1e6:.2f} µCi")
print("=====================================================================")

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.barh(['¹⁴C', '⁴⁰K (β⁻)', '⁴⁰K (CE + γ)'], [A_C_persona, 0.892 * A_K_persona, 0.107 * A_K_persona],
                 color=['#55a868', '#4c72b0', '#8172b3'])
ax.bar_label(barras, fmt='%.0f Bq', padding=3)
ax.set_xlabel('Actividad (Bq)'); ax.set_xlim(0, 4700)
ax.set_title('Radiactividad natural de una persona de 70 kg', fontweight='bold')
plt.tight_layout(); plt.show()

### **Problema 11**
El potasio natural contiene los isótopos 39, 40 y 41 en proporciones (expresadas en átomos por átomo de potasio natural) de 0,933, $1,19\cdot10^{-4}$ y 0,067 respectivamente. Los isótopos 39 y 41 son estables mientras que el $^{40}$K se desintegra por emisión $\beta^-$ convirtiéndose en el estado fundamental del $^{40}$Ca o por captura electrónica pasando a un estado excitado del $^{40}$Ar que decae al fundamental por emisión de un rayo $\gamma$. Los núcleos $^{40}$Ca y $^{40}$Ar son estables. Sabiendo que 1 gramo de potasio natural emite 28 partículas $\beta$ y 3,4 rayos $\gamma$ por segundo:

**a)** Hallar el periodo del $^{40}$K.

**b)** Hallar la edad de un mineral de potasio en el que hay acumulados $1,54\cdot10^{-2}$ cm³ (en condiciones normales) de argón por gramo de potasio.

---

#### **a) Periodo del $^{40}$K**
Peso atómico medio: $P_a = 0.933\times39 + 1.19\times10^{-4}\times40 + 0.067\times41 = 39.14$ g/mol.
$$ N(^{40}\text{K}) = 1.19\times10^{-4}\times\frac{6.02\times10^{23}}{39.14} = 1.83\times10^{18}\text{ núcleos por g de K} $$
Cada rama tiene su constante de desintegración parcial:
$$ \lambda_\beta = \frac{28}{1.83\times10^{18}} = 1.53\times10^{-17}\text{ s}^{-1}, \qquad \lambda_\gamma = \frac{3.4}{1.83\times10^{18}} = 1.86\times10^{-18}\text{ s}^{-1} $$
$$ \lambda = \lambda_\beta + \lambda_\gamma = 1.72\times10^{-17}\text{ s}^{-1} \quad\Rightarrow\quad T = \frac{\ln 2}{\lambda} = 4.03\times10^{16}\text{ s} = \mathbf{1.28\times10^9\text{ años}} $$
(La fracción de captura electrónica es $\lambda_\gamma/\lambda = 0.108$.)

#### **b) Datación K-Ar**
Si $N_K(t)$ es el número actual de núcleos de $^{40}$K, los que había al formarse el mineral eran $N_K e^{\lambda t}$. Los desintegrados se reparten entre Ar y Ca en la proporción de las constantes parciales:
$$ N_{Ar} = \frac{\lambda_\gamma}{\lambda}\,N_K\left(e^{\lambda t} - 1\right) \quad\Longrightarrow\quad \boxed{t = \frac{1}{\lambda}\ln\left(1 + \frac{\lambda}{\lambda_\gamma}\frac{N_{Ar}}{N_K}\right)} $$
Átomos de argón (un mol de gas ideal en condiciones normales ocupa 22400 cm³):
$$ N_{Ar} = \frac{1.54\times10^{-2}}{22400}\times6.02\times10^{23} = 4.14\times10^{17}\text{ átomos por g de K} $$
$$ t = 1.85\times10^9\text{ años}\times\ln\left(1 + \frac{4.14\times10^{17}}{0.108\times1.83\times10^{18}}\right) = \mathbf{2.08\times10^9\text{ años}} $$
El argón es un gas noble que no forma parte de la red cristalina: todo el que se encuentra atrapado en el mineral procede de la desintegración del $^{40}$K (el método supone que el mineral no ha perdido argón).

In [ ]:
# =====================================================================
# 1. DATOS DE ENTRADA
# =====================================================================
abund = {39: 0.933, 40: 1.19e-4, 41: 0.067}
emision_beta, emision_gamma = 28.0, 3.4      # por g de K y segundo
V_Ar = 1.54e-2                                # cm³ (c.n.) por g de K
V_mol = 22400.0                               # cm³/mol

# =====================================================================
# 2. CÁLCULOS
# =====================================================================
Pa_Knat = sum(A * f for A, f in abund.items())
N_K40 = abund[40] * NA / Pa_Knat
l_b, l_g = emision_beta / N_K40, emision_gamma / N_K40
l_K = l_b + l_g
T_K = np.log(2) / l_K
N_Ar = V_Ar / V_mol * NA
edad_KAr = np.log(1 + (l_K / l_g) * N_Ar / N_K40) / l_K

print("=====================================================================")
print("                RESULTADOS NUMÉRICOS (PROBLEMA 11)                   ")
print("=====================================================================")
print(f"a) P_a(K) = {Pa_Knat:.2f} g/mol,  N(40K) = {N_K40:.3e} por g de K")
print(f"   λ_β = {l_b:.3e} s⁻¹,  λ_γ = {l_g:.3e} s⁻¹,  λ = {l_K:.3e} s⁻¹  (fracción CE = {l_g/l_K:.3f})")
print(f"   T(40K) = {T_K:.3e} s = {T_K/anio:.3e} años")
print(f"b) N(Ar) = {N_Ar:.3e} átomos por g de K")
print(f"   Edad del mineral = {edad_KAr/anio:.3e} años")
print("=====================================================================")

# Reloj K-Ar: argón acumulado frente a la edad
t = np.linspace(0, 4.6e9, 400) * anio
NAr_t = (l_g / l_K) * N_K40 * (np.exp(l_K * t) - 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t / anio / 1e9, NAr_t / NA * V_mol, color='#4c72b0', lw=2.5)
ax.plot(edad_KAr / anio / 1e9, V_Ar, '*', color='gold', ms=18, mec='k', label=f'Mineral: {edad_KAr/anio/1e9:.2f}·10⁹ años')
ax.axvline(4.54, color='gray', ls=':'); ax.text(4.0, 0.002, 'Edad de\nla Tierra', color='gray')
ax.set_xlabel('Edad del mineral (10⁹ años)'); ax.set_ylabel('Argón acumulado (cm³ c.n. por g de K)')
ax.set_title('Reloj potasio-argón (con la abundancia actual de ⁴⁰K)', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

---
### **Resumen de la Hoja 4**

| Concepto | Expresión | Problemas |
| :--- | :--- | :--- |
| Ley de desintegración | $N = N_0e^{-\lambda t}$, $A = \lambda N$, $\lambda = \ln2/T$ | 1, 2 |
| Cadena A → B (Bateman) | $A_B = A_A(0)\frac{\lambda_B}{\lambda_B-\lambda_A}\left(e^{-\lambda_At} - e^{-\lambda_Bt}\right)$ | 2, 3, 4 |
| Equilibrio secular / transitorio | $A_B = A_A$ / $A_B/A_A = T_A/(T_A - T_B)$ | 2, 3, 4 |
| Conversión interna | $P_\gamma = 1/(1+\alpha)$, $P_{e,i} = \alpha_i/(1+\alpha)$ | 4, 5 |
| Activación neutrónica | $A = N\sigma\phi\left(1 - e^{-\lambda t_{irr}}\right)e^{-\lambda t_{enfr}}$ | 5, 6, 7, 8 |
| Autoapantallamiento | $N\sigma d \to \frac{\sigma_a}{\sigma_T}\left(1 - e^{-N\sigma_T d}\right)$ | 7 |
| Ramas de desintegración | $\lambda = \sum_i \lambda_i$ | 10, 11 |
| Datación | $t = \frac{1}{\lambda}\ln\frac{A_v}{A_m}$ (¹⁴C), $t = \frac{1}{\lambda}\ln\left(1 + \frac{\lambda}{\lambda_{CE}}\frac{N_{Ar}}{N_K}\right)$ (K-Ar) | 9, 11 |